# Named runtime repair: proof before production TDD

This notebook tests candidate routing and execution-origin contracts before production edits. Rust cells exercise a source-grounded policy model, not a compiled replacement for the host. Python cells run real ipykernel background work, then test a proposed context carrier with an explicit host receipt model. Production tests must reproduce the actual APIs and remain authoritative for RED/GREEN.

The repairs preserve factory kernels, notebook/profile ownership, current process registration, generation, source/version and Running admission. No copied factory port helper or filesystem publication bypass is used. After these proofs, production changes will be made in isolated external-repository worktrees.

In [3]:
mod routing_proof {
    pub fn current(spec: &str, slot: &str) -> bool {
        matches!(spec, "xcpp" | "xcpp17" | "xcpp20" | "xcpp23" | "evcxr")
            || (spec == "python3" && slot.starts_with("native-profile:"))
    }
    pub fn candidate(spec: &str, slot: &str) -> bool {
        matches!(spec, "xcpp" | "xcpp17" | "xcpp20" | "xcpp23" | "evcxr")
            || (matches!(spec, "python3" | "deno") && slot.starts_with("native-profile:"))
    }
}
let routing_cases = [
    ("deno", "native-profile:v1:valid-owned-profile", false, true),
    ("deno", "notebook:example.ipynb#deno", false, false),
    ("python3", "native-profile:v1:valid-owned-profile", true, true),
    ("python3", "notebook:example.ipynb#python3", false, false),
    ("evcxr", "notebook:example.ipynb#evcxr", true, true),
    ("xcpp20", "notebook:example.ipynb#xcpp20", true, true),
    ("unknown", "native-profile:v1:untrusted", false, false),
];
for (spec, slot, before, after) in routing_cases {
    assert_eq!(routing_proof::current(spec, slot), before);
    assert_eq!(routing_proof::candidate(spec, slot), after);
    println!("{} | {} | current={} candidate={}", spec, slot, before, after);
}
println!("ROUTING_MODEL_PASS: named Deno changes; factory Deno stays legacy. Ownership is validated later by authoritative dispatch.");

deno | native-profile:v1:valid-owned-profile | current=false candidate=true


deno | notebook:example.ipynb#deno | current=false candidate=false


python3 | native-profile:v1:valid-owned-profile | current=true candidate=true


python3 | notebook:example.ipynb#python3 | current=false candidate=false


evcxr | notebook:example.ipynb#evcxr | current=true candidate=true


xcpp20 | notebook:example.ipynb#xcpp20 | current=true candidate=true


unknown | native-profile:v1:untrusted | current=false candidate=false


ROUTING_MODEL_PASS: named Deno changes; factory Deno stays legacy. Ownership is validated later by authoritative dispatch.


In [2]:
import asyncio,threading,contextvars,json,os,sys
from dataclasses import dataclass,asdict
@dataclass(frozen=True)
class Origin:
 notebook: str
 profile: str
 process_generation: int
 source_version: int
 request_id: str
origin_var=contextvars.ContextVar('proof_execution_origin',default=None)
proof_kernel=get_ipython().kernel
origin_A=Origin('proof-notebook','proof-python',1,1,proof_kernel.get_parent()['header']['msg_id'])
origin_var.set(origin_A)
proof_async_gate=asyncio.Event(); proof_thread_gate=threading.Event()
proof_records={}
async def proof_async_worker():
 await proof_async_gate.wait()
 proof_records['async_origin']=origin_var.get()
 proof_records['async_parent']=proof_kernel.get_parent()['header']['msg_id']
def proof_thread_worker():
 proof_thread_gate.wait()
 proof_records['thread_origin']=origin_var.get()
 proof_records['thread_parent']=proof_kernel.get_parent()['header']['msg_id']
proof_task=asyncio.create_task(proof_async_worker())
proof_thread=threading.Thread(target=proof_thread_worker,daemon=True);proof_thread.start()
print(json.dumps({'phase':'A','origin':asdict(origin_A),'pid':os.getpid(),'executable':sys.executable,'scope':'carrier experiment; no port publication attempted'}))

{"phase": "A", "origin": {"notebook": "proof-notebook", "profile": "proof-python", "process_generation": 1, "source_version": 1, "request_id": "f12219f2-ff9b-41b3-9965-413c30b80d5e"}, "pid": 6865, "executable": "/Users/kevintruong/.spur/jupyter/venv-duckdb-2.0/bin/python", "scope": "carrier experiment; no port publication attempted"}


In [3]:
origin_B=Origin('proof-notebook','proof-python',1,2,proof_kernel.get_parent()['header']['msg_id'])
origin_var.set(origin_B)
proof_async_gate.set();proof_thread_gate.set()
await proof_task
proof_thread.join()
def model_admit(carrier,current_running):
 return carrier is not None and carrier==current_running
carrier_observation={
 'phase_A':asdict(origin_A),'phase_B':asdict(origin_B),
 'async_origin_preserved':proof_records['async_origin']==origin_A,
 'unbound_thread_has_no_origin':proof_records['thread_origin'] is None,
 'thread_latest_parent_is_B':proof_records['thread_parent']==origin_B.request_id,
 'current_B_admitted':model_admit(origin_var.get(),origin_B),
 'late_async_A_rejected_while_B_running':not model_admit(proof_records['async_origin'],origin_B),
 'unbound_thread_rejected':not model_admit(proof_records['thread_origin'],origin_B)}
assert all(carrier_observation[k] for k in ['async_origin_preserved','unbound_thread_has_no_origin','thread_latest_parent_is_B','current_B_admitted','late_async_A_rejected_while_B_running','unbound_thread_rejected'])
print(json.dumps(carrier_observation,indent=2))

{
  "phase_A": {
    "notebook": "proof-notebook",
    "profile": "proof-python",
    "process_generation": 1,
    "source_version": 1,
    "request_id": "f12219f2-ff9b-41b3-9965-413c30b80d5e"
  },
  "phase_B": {
    "notebook": "proof-notebook",
    "profile": "proof-python",
    "process_generation": 1,
    "source_version": 2,
    "request_id": "fecd0948-eb85-42b8-bd2f-f16c04c7efbb"
  },
  "async_origin_preserved": true,
  "unbound_thread_has_no_origin": true,
  "thread_latest_parent_is_B": true,
  "current_B_admitted": true,
  "late_async_A_rejected_while_B_running": true,
  "unbound_thread_rejected": true
}


In [4]:
mod publication_proof {
 #[derive(Clone,Debug,PartialEq,Eq)]
 pub struct Stamp {pub notebook:&'static str,pub profile:&'static str,pub generation:u64,pub source_version:u64,pub run:&'static str,pub fingerprint:&'static str,pub registration:&'static str}
 pub fn admit(language:&str,qualified:bool,carrier:Option<&Stamp>,current:&Stamp,running:bool,cancelled:bool)->bool {
   matches!(language,"deno"|"python") && qualified && running && !cancelled && carrier==Some(current)
 }
}
use publication_proof::{Stamp,admit};
let current=Stamp{notebook:"notebook-A",profile:"profile-A",generation:1,source_version:2,run:"run-B",fingerprint:"inputs-B",registration:"registration-B"};
assert!(admit("python",true,Some(&current),&current,true,false));
assert!(admit("deno",true,Some(&current),&current,true,false));
assert!(!admit("python",false,Some(&current),&current,true,false));
assert!(!admit("python",true,None,&current,true,false));
assert!(!admit("python",true,Some(&current),&current,false,false));
assert!(!admit("python",true,Some(&current),&current,true,true));
let mut invalid=Vec::new();
let mut x=current.clone();x.notebook="notebook-other";invalid.push(x);
let mut x=current.clone();x.profile="profile-other";invalid.push(x);
let mut x=current.clone();x.generation=2;invalid.push(x);
let mut x=current.clone();x.source_version=1;invalid.push(x);
let mut x=current.clone();x.run="run-A";invalid.push(x);
let mut x=current.clone();x.fingerprint="inputs-A";invalid.push(x);
let mut x=current.clone();x.registration="registration-A";invalid.push(x);
for stamp in &invalid { assert!(!admit("python",true,Some(stamp),&current,true,false)); }
println!("PUBLICATION_MODEL_PASS: both qualified languages admitted; {} identity mutations, absent carrier, stale Running, cancellation and missing qualification rejected. No production host IPC is exercised.",invalid.len());

PUBLICATION_MODEL_PASS: both qualified languages admitted; 7 identity mutations, absent carrier, stale Running, cancellation and missing qualification rejected. No production host IPC is exercised.


In [4]:
from pathlib import Path
import hashlib,subprocess
source_root=Path('/Users/kevintruong/project/spur-notebook')
source_files=['src/dag/engine.rs','src/mcp/tools/native_profile_lifecycle.rs','jute-notebook/src-tauri/src/kernel_config/target.rs','jute-notebook/src-tauri/src/kernel_config/runtime.rs','jute-notebook/src-tauri/src/kernel_config/runtime_provenance.rs','src/dag/deno_publication_bridge.rs']
proof_sources={'head':subprocess.run(['git','-C',str(source_root),'rev-parse','HEAD'],text=True,capture_output=True,check=True).stdout.strip(),'files':{p:hashlib.sha256((source_root/p).read_bytes()).hexdigest() for p in source_files}}
print(json.dumps(proof_sources,indent=2))

{
  "head": "b5885ef86b939680b7803c236ee81005beac8aa9",
  "files": {
    "src/dag/engine.rs": "3cf79ef72af674c326ec95897c21a53987046423b4246cf305e00830fa379d43",
    "src/mcp/tools/native_profile_lifecycle.rs": "a713df60c13498bad45b1f00cc9ca23eac44bad3100456ea82576837cd1af902",
    "jute-notebook/src-tauri/src/kernel_config/target.rs": "bd5a62dba8d4ae08b5b86e90019a6d11868c06c2f6c04206687b10f2aab2674e",
    "jute-notebook/src-tauri/src/kernel_config/runtime.rs": "35f210beefd47e901cb2d92d5b6b6aadea8686ec814b9e96bcf2343233e3148e",
    "jute-notebook/src-tauri/src/kernel_config/runtime_provenance.rs": "46b7176dab1191b9dd42caf17f92a6125ed7096a2481a2f8140d8732c9617643",
    "src/dag/deno_publication_bridge.rs": "75ad16df11c8072589f218fa760c6a94e6f02a3313e9fea031433e33392f1e11"
  }
}
